In [ ]:
import torch
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from lorentz import Lorentz

# ── config (change these if needed) ──────────────────────────────────────────
CKPT   = "ckpt/final_my_graph.ckpt"
VOCAB  = "enoa_vocab.pkl"
MATRIX = "enoa_graph.npy"
DIM    = 2
# ─────────────────────────────────────────────────────────────────────────────

# load vocab, matrix and model — shared across all cells below
genres = pickle.load(open(VOCAB, 'rb'))
n_items = len(genres)
adj = np.load(MATRIX)
genre_to_idx = {g: i for i, g in enumerate(genres)}

net = Lorentz(n_items, DIM + 1)
net.load_state_dict(torch.load(CKPT, map_location='cpu'))
net.eval()

table = net.lorentz_to_poincare()
coords = table[1:]
x = coords[:, 0]
y = coords[:, 1]

print(f'Loaded {n_items} genres and embeddings')

In [ ]:
# ── Option 1: Top N most connected genres ────────────────────────────────────
TOP_N    = 50
FIGSIZE  = (20, 20)
FONTSIZE = 8

degrees = (adj > 0).sum(axis=1)
top_indices = set(np.argsort(degrees)[-TOP_N:])
print(f'Top {TOP_N} genres:')
print([genres[i] for i in sorted(top_indices, key=lambda i: degrees[i], reverse=True)])

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
ax.add_artist(plt.Circle((0, 0), 1, fill=False, edgecolor='black', linewidth=1.5))

# all dots in grey
ax.scatter(x, y, s=8, color='lightgrey', zorder=2, linewidths=0)

# top N in red with labels
top_x = [x[i] for i in top_indices]
top_y = [y[i] for i in top_indices]
ax.scatter(top_x, top_y, s=30, color='tomato', zorder=4, linewidths=0)

for idx in top_indices:
    ax.annotate(
        genres[idx], (x[idx], y[idx]),
        fontsize=FONTSIZE, ha='center', va='bottom',
        xytext=(0, 4), textcoords='offset points', zorder=5,
        bbox={'fc': 'white', 'alpha': 0.85, 'edgecolor': 'none', 'pad': 1},
    )

ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-1.05, 1.05)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title(f'Poincaré Disk — Top {TOP_N} Most Connected Genres', fontsize=16, pad=15)
plt.tight_layout()
plt.savefig('plot_top_genres.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved to plot_top_genres.png')

In [ ]:
# ── Option 2: Known genre families ───────────────────────────────────────────
FIGSIZE  = (20, 20)
FONTSIZE = 9

FAMILIES = {
    'Latin':      {'genres': ['latin', 'reggaeton', 'cumbia', 'bachata', 'latin pop',
                               'trap latino', 'latin hip hop', 'reggaeton flow'],
                   'color': '#e63946'},
    'Hip Hop':    {'genres': ['rap', 'trap', 'hip hop', 'pop rap', 'melodic rap',
                               'gangster rap', 'east coast hip hop', 'southern hip hop'],
                   'color': '#f4a261'},
    'Rock':       {'genres': ['rock', 'metal', 'blues rock', 'indie rock', 'hard rock',
                               'alternative rock', 'classic rock', 'punk'],
                   'color': '#2a9d8f'},
    'Electronic': {'genres': ['edm', 'house', 'electropop', 'progressive house',
                               'tropical house', 'deep house', 'techno', 'dance pop'],
                   'color': '#457b9d'},
    'Pop':        {'genres': ['pop', 'dance pop', 'post-teen pop', 'indie pop',
                               'art pop', 'synthpop', 'k-pop', 'j-pop'],
                   'color': '#a8dadc'},
}

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
ax.add_artist(plt.Circle((0, 0), 1, fill=False, edgecolor='black', linewidth=1.5))

# all dots in light grey
ax.scatter(x, y, s=8, color='lightgrey', zorder=2, linewidths=0)

legend_patches = []
for family_name, info in FAMILIES.items():
    color = info['color']
    found_indices = [genre_to_idx[g] for g in info['genres'] if g in genre_to_idx]
    found_genres  = [g for g in info['genres'] if g in genre_to_idx]
    missing = [g for g in info['genres'] if g not in genre_to_idx]
    if missing:
        print(f'{family_name} — missing: {missing}')

    if not found_indices:
        continue

    ax.scatter([x[i] for i in found_indices], [y[i] for i in found_indices],
               s=60, color=color, zorder=4, linewidths=0)

    for i, genre in zip(found_indices, found_genres):
        ax.annotate(
            genre, (x[i], y[i]),
            fontsize=FONTSIZE, ha='center', va='bottom',
            xytext=(0, 5), textcoords='offset points', zorder=5,
            color=color, fontweight='bold',
            bbox={'fc': 'white', 'alpha': 0.85, 'edgecolor': 'none', 'pad': 1},
        )

    legend_patches.append(mpatches.Patch(color=color, label=family_name))

ax.legend(handles=legend_patches, loc='lower right', fontsize=12, framealpha=0.9)
ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-1.05, 1.05)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Poincaré Disk — Genre Families', fontsize=16, pad=15)
plt.tight_layout()
plt.savefig('plot_genre_families.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved to plot_genre_families.png')